# TIỀN XỬ LÝ & KIỂM TRA CHẤT LƯỢNG DỮ LIỆU - TECHX ACADEMY

## 📌 BỐI CẢNH DỰ ÁN

* TechX Academy thành lập 2015 - trường học công nghệ tại Việt Nam.
* Sản phẩm chính:
    - Khóa IT Fullstack                     (Full time)
    - Khóa IT Bachelor's Program
    - Khóa lập trình viên Nhật Bản          (Fukuoka)
    - Khóa IT Developer Parttime            (Part time)
    - Khóa lập trình Web
    - Khóa Data Analytics Foundation        (Data Analyst)
* Mục tiêu: Đánh giá thực trạng hiệu quả kinh doanh 2022-2024,
  phân tích theo dòng sản phẩm | tư vấn viên | nguồn khách hàng.
* Dữ liệu đầu vào: Courses of Interest Table.xlsx (Sheet1)

## 🛠️ PIPELINE

CELL 1  → Import thư viện & cấu hình
CELL 2  → Đọc dữ liệu
CELL 3  → Kiểm tra tổng quan
CELL 4  → Làm sạch cơ bản
CELL 5  → Chuyển cột ngày tháng
CELL 6  → Xử lý hieu_suat (serial date vs số ngày)
CELL 7  → Tính số ngày tư vấn → chốt sale
CELL 8  → Chuẩn hóa cột phân loại
CELL 9  → ⚠️  KIỂM TRA quan_tam_khoa_hoc ↔ quan_tam_khoa_hoc_1
CELL 10 → ⚠️  KIỂM TRA LỖI LOGIC KHÁC
CELL 11 → Tách khách hàng & tư vấn viên từ ID
CELL 12 → Báo cáo chất lượng dữ liệu cuối
CELL 13 → Xuất file sạch + file báo cáo lỗi


In [1]:
import pandas as pd
import numpy as np
import os

INPUT_FILE   = "Courses of Interest Table.xlsx"
OUTPUT_XLSX  = "Courses_of_Interest_Cleaned.xlsx"
OUTPUT_CSV   = "Courses_of_Interest_Cleaned.csv"
ISSUES_FILE  = "Data_Quality_Issues.xlsx"
SHEET_NAME   = "Sheet1"

def print_header(title):
    print("\n" + "=" * 72)
    print(f"  {title}")
    print("=" * 72)

print_header("CELL 1: IMPORT & CẤU HÌNH")
print("✅ KẾT LUẬN CELL 1: Sẵn sàng pipeline với 3 file output")


  CELL 1: IMPORT & CẤU HÌNH
✅ KẾT LUẬN CELL 1: Sẵn sàng pipeline với 3 file output


In [2]:
print_header("CELL 2: ĐỌC DỮ LIỆU")

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"Không tìm thấy file: {INPUT_FILE}")

df = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME)
print(f"Kích thước ban đầu: {df.shape[0]} dòng x {df.shape[1]} cột")
print("\n5 dòng đầu:")
print(df.head().to_string())

print("✅ KẾT LUẬN CELL 2: Đã đọc thành công ~10.999 dòng")


  CELL 2: ĐỌC DỮ LIỆU
Kích thước ban đầu: 18978 dòng x 13 cột

5 dòng đầu:
   STT                              ID quan_tam_khoa_hoc          ngay_tao  tao_lich_tu_van thoi_gian_chuyen_sang_tu_van thoi_gian_chuyen_sang_chot_sale  hieu_suat   MKT_xu_ly_data nha_mang check_zalo quan_tam_khoa_hoc_1 sentimatic
0    0      Chu Chiến-Trương Thị Huyền         Full time  2023/12/29 13:57  Tạo lịch tư vấn                            0                               0          0  Chăm sóc vòng 1     Vina   Xem Zalo              Khóa A   tieu_cuc
1    1  Xuân Sáng-Trịnh Thị Thùy Dương         Full time  2023/12/29 14:01  Tạo lịch tư vấn                            0                               0          0  Chăm sóc vòng 1  Viettel   Xem Zalo              Khóa A   tieu_cuc
2    2          Huy Cuong-Phạm Mỹ Linh         Part time  2023/12/29 14:03  Tạo lịch tư vấn                            0                               0          0  Chăm sóc vòng 1  Viettel   Xem Zalo              Khóa B   tieu_

In [3]:
print_header("CELL 3: KIỂM TRA TỔNG QUAN")

print("\n--- Kiểu dữ liệu ---")
print(df.dtypes)

print("\n--- Missing gốc ---")
print(df.isna().sum())

print("\n--- Số dòng trùng hoàn toàn ---")
print(df.duplicated().sum())

print("✅ KẾT LUẬN CELL 3:\n  • STT, ngày tháng đang ở dạng object\n  • hieu_suat trộn serial date và số ngày\n  • Có nhiều giá trị 0 thay cho missing")


  CELL 3: KIỂM TRA TỔNG QUAN

--- Kiểu dữ liệu ---
STT                                 int64
ID                                    str
quan_tam_khoa_hoc                  object
ngay_tao                              str
tao_lich_tu_van                    object
thoi_gian_chuyen_sang_tu_van       object
thoi_gian_chuyen_sang_chot_sale    object
hieu_suat                           int64
MKT_xu_ly_data                     object
nha_mang                           object
check_zalo                         object
quan_tam_khoa_hoc_1                object
sentimatic                            str
dtype: object

--- Missing gốc ---
STT                                0
ID                                 0
quan_tam_khoa_hoc                  0
ngay_tao                           0
tao_lich_tu_van                    0
thoi_gian_chuyen_sang_tu_van       0
thoi_gian_chuyen_sang_chot_sale    0
hieu_suat                          0
MKT_xu_ly_data                     0
nha_mang                          

In [4]:
print_header("CELL 4: LÀM SẠCH CƠ BẢN")

df.columns = df.columns.str.strip()
df["STT"] = pd.to_numeric(df["STT"], errors="coerce")

before = len(df)
df = df[df["STT"].notna()].copy()
df["STT"] = df["STT"].astype(int)
print(f"Đã loại {before - len(df)} dòng STT lỗi.")


  CELL 4: LÀM SẠCH CƠ BẢN
Đã loại 0 dòng STT lỗi.


In [ ]:
print_header("CELL 4: THAY GIÁ TRỊ 0 BẰNG NaN")

zero_cols = [
    "quan_tam_khoa_hoc",
    "thoi_gian_chuyen_sang_tu_van",
    "thoi_gian_chuyen_sang_chot_sale",
    "hieu_suat",
    "quan_tam_khoa_hoc_1",
]
for col in zero_cols:
    if col in df.columns:
        df[col] = df[col].replace([0, "0", ""], np.nan)

print(f"Kích thước sau làm sạch: {df.shape}")
print("Đã thay giá trị 0 bằng NaN.")


  CELL 4: THAY GIÁ TRỊ 0 BẰNG NaN
Kích thước sau làm sạch: (18978, 13)
Đã thay giá trị 0 bằng NaN.


: 